# Approximate Unlearning

## What This Is
Approximate unlearning modifies a trained model to remove the influence of specific data points without full retraining. Three main approaches:

1. **Gradient ascent**: Take gradient steps in the direction that *increases* loss on the forget set, reversing the learning signal
2. **Newton step (influence functions)**: Use the Hessian to compute exactly how weights would change if the forget set had never been included
3. **Fine-tuning on retain set**: Continue training only on the retain set until the forget set's influence decays

## Why Not Just Gradient Ascent?
Naive gradient ascent unlearning is unstable — ascending on the forget set can degrade model performance on the retain set catastrophically. Careful regularization (e.g., keeping parameters close to original) or alternating ascent/descent steps are necessary.

In [ ]:
import numpy as np
import copy

np.random.seed(42)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def logistic_loss(X, y, w, b):
    z = X @ w + b
    probs = sigmoid(z)
    probs = np.clip(probs, 1e-10, 1 - 1e-10)
    return -np.mean(y * np.log(probs) + (1 - y) * np.log(1 - probs))

def logistic_grad(X, y, w, b):
    n = len(y)
    z = X @ w + b
    err = sigmoid(z) - y
    dw = (X.T @ err) / n
    db = err.mean()
    return dw, db

def train_logistic(X, y, lr=0.1, epochs=200):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    for _ in range(epochs):
        dw, db = logistic_grad(X, y, w, b)
        w -= lr * dw
        b -= lr * db
    return w, b

def accuracy(X, y, w, b):
    return (( sigmoid(X @ w + b) > 0.5).astype(int) == y).mean()

# Rebuild dataset
N, D = 500, 10
X_all = np.random.randn(N, D)
true_w = np.random.randn(D)
y_all = (sigmoid(X_all @ true_w) > 0.5).astype(float)
forget_idx = list(range(50, 70))
retain_idx = [i for i in range(N) if i not in forget_idx]
X_retain, y_retain = X_all[retain_idx], y_all[retain_idx]
X_forget, y_forget = X_all[forget_idx], y_all[forget_idx]

w_orig, b_orig = train_logistic(X_all, y_all)
w_retrain, b_retrain = train_logistic(X_retain, y_retain)
print('Baseline models trained.')
print(f'Original   - retain acc: {accuracy(X_retain, y_retain, w_orig, b_orig):.3f}')
print(f'Retrained  - retain acc: {accuracy(X_retain, y_retain, w_retrain, b_retrain):.3f}')


In [ ]:
# Method 1: Gradient Ascent Unlearning
# Ascend on forget set loss + regularize toward original weights to prevent collapse

def gradient_ascent_unlearn(w_init, b_init, X_forget, y_forget, 
                             X_retain, y_retain,
                             lr_ascent=0.05, lr_descent=0.02,
                             epochs=50, lam=0.1):
    w = w_init.copy()
    b = b_init.copy() if hasattr(b_init, 'copy') else b_init
    w0 = w_init.copy()  # anchor
    
    for ep in range(epochs):
        # Ascend on forget set (maximize loss = minimize memory)
        dw_f, db_f = logistic_grad(X_forget, y_forget, w, b)
        # Descend on retain set (preserve utility)
        dw_r, db_r = logistic_grad(X_retain, y_retain, w, b)
        # Regularize toward original (prevent drift)
        reg_w = lam * (w - w0)
        
        w = w + lr_ascent * dw_f - lr_descent * dw_r - 0.01 * reg_w
        b = b + lr_ascent * db_f - lr_descent * db_r
    
    return w, b

w_ga, b_ga = gradient_ascent_unlearn(
    w_orig, b_orig, X_forget, y_forget, X_retain, y_retain
)

print('Gradient Ascent Unlearning Results:')
print(f'  Retain accuracy: {accuracy(X_retain, y_retain, w_ga, b_ga):.3f} '
      f'(orig: {accuracy(X_retain, y_retain, w_orig, b_orig):.3f}, '
      f'retrain: {accuracy(X_retain, y_retain, w_retrain, b_retrain):.3f})')
print(f'  Weight dist from retrained: {np.linalg.norm(w_ga - w_retrain):.4f}')
print(f'  Weight dist from original:  {np.linalg.norm(w_ga - w_orig):.4f}')


In [ ]:
# Method 2: Newton Step Unlearning (Influence Function approach)
# Uses second-order information to compute the exact weight change
# that would result from removing the forget set

def compute_hessian(X, y, w, b, lam=1e-3):
    """Hessian of logistic regression loss w.r.t. w."""
    z = X @ w + b
    p = sigmoid(z)
    D_diag = p * (1 - p)  # diagonal of the Hessian contribution
    H = (X.T * D_diag) @ X / len(y) + lam * np.eye(X.shape[1])
    return H

def newton_unlearn(w_orig, b_orig, X_all, y_all, X_forget, y_forget, lam=1e-3):
    """Remove forget set influence via Newton step."""
    n = len(y_all)
    nf = len(y_forget)
    
    # Gradient of forget set loss
    dw_f, db_f = logistic_grad(X_forget, y_forget, w_orig, b_orig)
    # Scale by forget set fraction
    scale = nf / n
    
    # Hessian of full dataset loss
    H = compute_hessian(X_all, y_all, w_orig, b_orig, lam)
    H_inv = np.linalg.inv(H)
    
    # Newton step: w_new = w_orig + H^{-1} * scale * grad_forget
    delta_w = H_inv @ (scale * dw_f)
    w_new = w_orig + delta_w
    b_new = b_orig + scale * db_f  # simplified for bias
    
    return w_new, b_new

w_newton, b_newton = newton_unlearn(
    w_orig, b_orig, X_all, y_all, X_forget, y_forget
)

print('Newton Step (Influence Function) Unlearning Results:')
print(f'  Retain accuracy: {accuracy(X_retain, y_retain, w_newton, b_newton):.3f}')
print(f'  Weight dist from retrained: {np.linalg.norm(w_newton - w_retrain):.4f}')
print(f'  Weight dist from original:  {np.linalg.norm(w_newton - w_orig):.4f}')


In [ ]:
# Comparison: all three methods
print('=' * 60)
print('Unlearning Method Comparison')
print('=' * 60)
methods = [
    ('Original (no unlearning)', w_orig, b_orig),
    ('Exact retrain (gold standard)', w_retrain, b_retrain),
    ('Gradient ascent', w_ga, b_ga),
    ('Newton step', w_newton, b_newton),
]

def mia_confidence(X, y, w, b):
    z = X @ w + b
    probs = sigmoid(z)
    return np.where(y == 1, probs, 1 - probs).mean()

print(f'{'Method':<35} {'Retain Acc':>10} {'Forget MIA':>11} {'W Dist Retrain':>15}')
print('-' * 75)
for name, w, b in methods:
    racc = accuracy(X_retain, y_retain, w, b)
    fmia = mia_confidence(X_forget, y_forget, w, b)
    wdist = np.linalg.norm(w - w_retrain)
    print(f'{name:<35} {racc:>10.3f} {fmia:>11.3f} {wdist:>15.4f}')

print('\nLower Forget MIA score = better unlearning (less membership signal)')
print('Lower Weight Dist from Retrain = closer to gold standard')
